In [1]:
import fitz 
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [2]:

import os
from dotenv import load_dotenv
load_dotenv()


os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [90]:
### Embedding functions
import torch.nn.functional as F


def embed_image(image_data):
    """
                                                                                Embed image using CLIP
    """
    if isinstance(image_data, str): 
        image = Image.open(image_data).convert("RGB")
    else: 
        image = image_data
    
    inputs=clip_processor(images=image,return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        # if huggingface returns object instead of tensor
        if not torch.is_tensor(features):
            features = features[0]

        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()
    
def embed_text(text):
    """Embed text using CLIP."""
    inputs = clip_processor(
        text=text, 
        return_tensors="pt", 
        padding=True,
        truncation=True,
        max_length=77  
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)

        # if huggingface returns object instead of tensor
        if not torch.is_tensor(features):
            features = features[0]

        features = features / features.norm(dim=-1, keepdim=True)

        return features.squeeze().numpy()



#testing
print(embed_text("A graph showing GDP growth"))

[[ 0.01234788  0.00423842  0.00371038 ...  0.00898103  0.02149546
   0.00368668]
 [ 0.0812392  -0.02403369  0.01515724 ...  0.04794865  0.03311041
  -0.04031156]
 [ 0.07442085 -0.08007197 -0.01482519 ... -0.00383499 -0.01968475
  -0.02036241]
 ...
 [ 0.02376101  0.00394842 -0.03483561 ... -0.03304572  0.00374539
  -0.07298761]
 [ 0.03236865 -0.02606504  0.00142386 ...  0.00947267  0.03574927
  -0.03914627]
 [ 0.00980587 -0.0697908   0.01708118 ...  0.02786418  0.02317797
  -0.04376754]]


In [105]:
## Process PDF


pdf_path="stats_qa.pdf"
doc=fitz.open(pdf_path)

all_docs = []
all_embeddings = []
image_data_store = {}  #


splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=500)




In [106]:
doc

Document('stats_qa.pdf')

In [107]:
for i,page in enumerate(doc):
    ## process text
    text=page.get_text()
    if text.strip():
        ##create temporary document for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        #Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)



    ## process images
    ## Three Step to do:

    ##Convert PDF image to PIL format
    ##Store as base64 for GPT-4V (which needs base64 images)
    ##Create CLIP embedding for retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"
            
            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)
            
            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()


In [108]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content="1/12\n\xa0You:\n------------------\n\xa0ChatGPT:\n1. Question: What is the Central Limit Theorem and why is it important?\nAnswer: The Central Limit Theorem (CLT) states that the distribution of the sum (or average) of a large number of independent, identically distributed\nrandom variables approaches a normal (or Gaussian) distribution, regardless of the original distribution of the variables. It's crucial in statistics\nbecause it allows us to make inferences about populations using the normal distribution, which has well-understood properties.\n2. Question: Explain Type I and Type II errors.\nAnswer:\nType I Error (False Positive, or Alpha error): It's when you incorrectly reject a true null hypothesis.\nType II Error (False Negative, or Beta error): It's when you fail to reject a false null hypothesis. The significance level (usually denoted by α) is\nthe probability of making a Type I error. The power of a test is 1 min

In [116]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array([emb.mean(axis=0) for emb in all_embeddings], dtype=np.float32)
embeddings_array.shape

(66, 512)

In [114]:
(all_docs,embeddings_array)

([Document(metadata={'page': 0, 'type': 'text'}, page_content="1/12\n\xa0You:\n------------------\n\xa0ChatGPT:\n1. Question: What is the Central Limit Theorem and why is it important?\nAnswer: The Central Limit Theorem (CLT) states that the distribution of the sum (or average) of a large number of independent, identically distributed\nrandom variables approaches a normal (or Gaussian) distribution, regardless of the original distribution of the variables. It's crucial in statistics\nbecause it allows us to make inferences about populations using the normal distribution, which has well-understood properties.\n2. Question: Explain Type I and Type II errors.\nAnswer:\nType I Error (False Positive, or Alpha error): It's when you incorrectly reject a true null hypothesis.\nType II Error (False Negative, or Beta error): It's when you fail to reject a false null hypothesis. The significance level (usually denoted by α) is\nthe probability of making a Type I error. The power of a test is 1 mi

In [117]:


# note: Create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,  # I have used precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [118]:
# Initialize GPT-4 Vision model
llm = init_chat_model("openai:gpt-4.1")
llm

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000016F84A84640>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000016F84A874F0>, root_client=<openai.OpenAI object at 0x0000016F84A85F60>, root_async_client=<openai.AsyncOpenAI object at 0x0000016F84A85870>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [123]:
def retrieve_multimodal(query, k=5):
    """
    Unified retrieval using CLIP embeddings for both text and images.
    """
    query_embedding = embed_text(query)  
    
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    
    return results

In [124]:
def create_multimodal_message(query, retrieved_docs):
    """
                                                                Create a message with both text and images for GPT-4V.
    """
    content = []
    

    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [125]:
def multimodal_pdf_rag_pipeline(query):
    """
                                                                        Main pipeline for multimodal RAG.
    """
    query_embedding = embed_text(query)

    # If query_embedding is 1D, reshape it to (1, embedding_dim)
    #  RCGP
    # Retrieve                     relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Create                    multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get                     response from GPT-4V
    response = llm.invoke([message])
    
    # Print                 retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    
  
    
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [126]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "Give me a summary of the 5 min quick revision notes of the document",
        "Give me the text highlighted in purple color",  
        "Give me the text highlighted in yellow color",
        "What visual elements are present in the document?"
    ] # I have highligted in purple color on 3 random pages
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: Give me a summary of the 5 min quick revision notes of the document
--------------------------------------------------


ValueError: too many values to unpack (expected 2)